In [2]:
from langchain_classic.retrievers import EnsembleRetriever,BM25Retriever
# from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores.chroma import Chroma
from langchain_core.documents import Document
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


c:\Users\pramo\OneDrive\Desktop\ChatBot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
chunks = [
    """Python is a high-level programming language widely used for automation, web development, machine learning, and data analysis. It emphasizes readability and has a large ecosystem of libraries.""",

    """FastAPI is a modern Python framework for building REST APIs. It automatically generates OpenAPI documentation and provides high performance using ASGI.""",

    """Docker packages applications into lightweight containers. Containers make applications portable and ensure they run consistently across development, testing, and production environments.""",

    """Kubernetes is a container orchestration platform that automates deployment, scaling, and management of containerized applications.""",

    """Retrieval-Augmented Generation (RAG) combines information retrieval with large language models. Relevant documents are retrieved from a vector database before generating a response.""",

    """Embeddings convert text into dense numerical vectors that capture semantic meaning. Similar vectors are located using cosine similarity or dot product.""",

    """BM25 is a sparse (lexical) retrieval algorithm used in information retrieval systems. It ranks documents based on exact keyword matching, term frequency (TF), inverse document frequency (IDF), and document length normalization. BM25 performs exceptionally well when user queries contain the same keywords as the indexed documents.""",

    """Hybrid Search combines dense vector retrieval with sparse keyword retrieval to achieve better search quality. Dense retrieval understands semantic meaning using embeddings, while sparse retrieval focuses on exact keyword matches. Combining both approaches improves recall and precision in Retrieval-Augmented Generation (RAG) systems.""",

    """Chroma is an open-source vector database used for storing embeddings and performing semantic similarity search. It enables efficient nearest-neighbor retrieval and integrates seamlessly with LangChain for building RAG applications.""",

    """LangChain is a framework for building LLM-powered applications. It provides abstractions for prompts, retrievers, agents, tools, memory, document loaders, output parsers, and workflows, making it easier to develop production-ready AI systems.""",

    """Transformers are deep learning architectures based on the self-attention mechanism. Unlike recurrent neural networks (RNNs), transformers process all tokens in parallel, enabling faster training and better handling of long-range dependencies.""",

    """Self-attention is the core mechanism behind transformer models. It allows every token in a sequence to attend to every other token, helping the model understand context, relationships, and dependencies across an entire sentence.""",

    """Git is a distributed version control system used to track changes in source code. GitHub is a cloud-based platform that hosts Git repositories and enables collaboration through pull requests, issues, branches, and code reviews.""",

    """SQL JOIN operations combine data from multiple database tables. INNER JOIN returns only matching records, LEFT JOIN returns all records from the left table, RIGHT JOIN returns all records from the right table, and FULL OUTER JOIN returns all matching and non-matching records.""",

    """Power BI is a business intelligence and data visualization platform developed by Microsoft. It connects to multiple data sources, transforms data using Power Query, creates interactive dashboards, and performs advanced calculations using DAX (Data Analysis Expressions)."""
]

In [ ]:
documents = [Document(page_content = chunk , metadata={"source":f"chunk_{i}"}) for i, chunk in enumerate(chunks)]
print("sample data")
for i , chunk in enumerate(documents):
    print(f"chunk {i} : {chunk.page_content}")
print()


In [5]:
#creating vector search
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
vectors = Chroma.from_documents(documents,embedding)

vector_retriever = vectors.as_retriever(search_kwargs={"k":3})
test_query = "vector retrieval "
res = vector_retriever.invoke(test_query)
for doc in res:
    print(f"chunk {doc.metadata['source']} : {doc.page_content}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4741.88it/s]


chunk chunk_7 : Hybrid Search combines dense vector retrieval with sparse keyword retrieval to achieve better search quality. Dense retrieval understands semantic meaning using embeddings, while sparse retrieval focuses on exact keyword matches. Combining both approaches improves recall and precision in Retrieval-Augmented Generation (RAG) systems.
chunk chunk_8 : Chroma is an open-source vector database used for storing embeddings and performing semantic similarity search. It enables efficient nearest-neighbor retrieval and integrates seamlessly with LangChain for building RAG applications.
chunk chunk_4 : Retrieval-Augmented Generation (RAG) combines information retrieval with large language models. Relevant documents are retrieved from a vector database before generating a response.


In [7]:
## keyword search using bm25
bm25 = BM25Retriever.from_documents(documents)
bm25.k = 3
test_query = "vector retrieval"
res = bm25.invoke(test_query)

for doc in res :
    print(f"chunk {doc.metadata['source']}:{doc.page_content}")


chunk chunk_7:Hybrid Search combines dense vector retrieval with sparse keyword retrieval to achieve better search quality. Dense retrieval understands semantic meaning using embeddings, while sparse retrieval focuses on exact keyword matches. Combining both approaches improves recall and precision in Retrieval-Augmented Generation (RAG) systems.
chunk chunk_4:Retrieval-Augmented Generation (RAG) combines information retrieval with large language models. Relevant documents are retrieved from a vector database before generating a response.
chunk chunk_8:Chroma is an open-source vector database used for storing embeddings and performing semantic similarity search. It enables efficient nearest-neighbor retrieval and integrates seamlessly with LangChain for building RAG applications.


In [ ]:
# HYBRIDSRARCH

hybrid_search = EnsembleRetriever(
    retrievers = [vector_retriever,bm25],
    weights=[0.5,0.5],
    k=3
)
# while test_query != "exit":
test_query = input("enter the query")
res = hybrid_search.invoke(test_query)


for doc in res:
    print(f"chunk {doc.metadata['source']} : {doc.page_content}")


chunk chunk_14 : Power BI is a business intelligence and data visualization platform developed by Microsoft. It connects to multiple data sources, transforms data using Power Query, creates interactive dashboards, and performs advanced calculations using DAX (Data Analysis Expressions).
chunk chunk_6 : BM25 is a sparse (lexical) retrieval algorithm used in information retrieval systems. It ranks documents based on exact keyword matching, term frequency (TF), inverse document frequency (IDF), and document length normalization. BM25 performs exceptionally well when user queries contain the same keywords as the indexed documents.
chunk chunk_13 : SQL JOIN operations combine data from multiple database tables. INNER JOIN returns only matching records, LEFT JOIN returns all records from the left table, RIGHT JOIN returns all records from the right table, and FULL OUTER JOIN returns all matching and non-matching records.
chunk chunk_9 : LangChain is a framework for building LLM-powered appli

In [33]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.1-8b-instant")
prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful assistant.

    Use ONLY the provided context to answer the user's question.

    If the answer is not present in the context, reply exactly:
    "I don't know."

    Give a concise answer.
    # Context: {context}
    Question: {question}
    """
)
def context(documents):
    return "\n\n".join(doc.page_content for doc in documents)

retriever = hybrid_search

doc_chain = prompt | llm | StrOutputParser()

rag_chain = {
    
    "context": retriever | RunnableLambda(context),
    "question": RunnablePassthrough(),
} | doc_chain

res = rag_chain.invoke("Why is hybrid search better than using only BM25?")
res



'Hybrid Search is better than using only BM25 because it combines the strengths of dense vector retrieval (understanding semantic meaning using embeddings) with the exact keyword matching capabilities of sparse keyword retrieval (like BM25), improving recall and precision in Retrieval-Augmented Generation (RAG) systems.'